In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
# first-time use execution
!pip install biopython

In [ ]:
!pip install scipy --target=$nb_path

In [ ]:
!pip install matplotlib ipywidgets --target=$nb_path

In [ ]:
!apt-get install -y python3-pymol

In [ ]:
## non-first-time execution
## package was previously installed through process_data_positive_patch_3points-Graph.ipynb
import sys
sys.path.append('/content/drive/MyDrive/@antibody/antigen_epitope/packages')

In [ ]:
import os
import numpy as np
import pandas as pd
from Bio import PDB

In [ ]:
# INSTALL pymesh!!
# upload the .whl file for installation
# https://github.com/nuvolos-cloud/PyMesh/releases
from google.colab import files
uploaded = files.upload()

## install pymesh to calculate the curvature of the protein surface
!pip install pymesh2-0.3.1-cp310-cp310-linux_x86_64.whl --target=$nb_path  --upgrade

In [ ]:
import pymesh

In [ ]:
#@title translate the geometric center of the protein to the origin (0, 0, 0),

from Bio import PDB

# Initialize a PDB parser
parser = PDB.PDBParser(QUIET=True)

# Load the structure
pdb_file_path = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_surface_index_yuhao/1a2y_B_A_C.pdb'
structure = parser.get_structure('1a2y_B_A_C', pdb_file_path)
# Calculate the geometric center (centroid) of the protein

sum_x, sum_y, sum_z = 0.0, 0.0, 0.0
n_atoms = 0
for model in structure:
    for chain in model:
        for residue in chain:
            for atom in residue:
                sum_x += atom.coord[0]
                sum_y += atom.coord[1]
                sum_z += atom.coord[2]
                n_atoms += 1

centroid = (sum_x / n_atoms, sum_y / n_atoms, sum_z / n_atoms)

# Translate each atom in the structure so that the centroid is moved to (0, 0, 0)
for model in structure:
    for chain in model:
        for residue in chain:
            for atom in residue:
                atom.set_coord(atom.coord - centroid)

# Save the modified structure to a new file
io = PDB.PDBIO()
io.set_structure(structure)
io.save('/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_surface_index_yuhao/1a2y_B_A_C_centered.pdb')

In [ ]:
#@title TEST generate mesh for all the atoms of pdb
import pandas as pd
import numpy as np
import pymesh
from Bio.PDB import PDBParser, PDBIO

from pymol import cmd


def get_surface_atoms(pdb_file_path, antigen_chain, threshold=0.2):
    """
    Use PyMOL to calculate SaSaRate and return atoms that are on the protein surface.
    """
    # Load the PDB file into PyMOL
    cmd.reinitialize()  # Clear previous data
    cmd.load(pdb_file_path, "protein_complex")

    # Calculate relative SASA
    SaSaRate = cmd.get_sasa_relative("protein_complex and chain " + antigen_chain)
    surface_residues = [key[1] for key, value in SaSaRate.items() if value > threshold]

    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('protein', pdb_file_path)
    model = structure[0]
    chain = model[antigen_chain]

    surface_atoms = []
    for residue in chain:
        if residue.id[1] in surface_residues:
            surface_atoms.extend(residue.get_atoms())

    return surface_atoms


def calculate_curvature(pdb_file_path, n_neighbors=3):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('protein', pdb_file_path)
    model = structure[0]
    chain = model['C']

    atoms = get_surface_atoms(pdb_file_path, 'C', threshold=0.2)

    #atoms = [atom for atom in chain.get_atoms()]
    coords = np.array([atom.coord for atom in atoms])
    curvatures = [None] * len(atoms)

    for idx, atom in enumerate(atoms):
        atom_coords = np.array(coords[idx])
        dist_coords = coords - atom_coords

        # Calculate distances and get indices of nearest neighbors
        dists = np.linalg.norm(dist_coords, axis=1)
        nearest_neighbors_idx = np.argsort(dists)[1:(n_neighbors + 1)]

        # Create vertices and faces
        vertices = dist_coords[np.concatenate([[idx], nearest_neighbors_idx]), :]
        faces = [[0, i, i + 1] for i in range(1, n_neighbors)] + [[0, n_neighbors, 1]]

        faces = np.array(faces)

        # Create mesh
        mesh = pymesh.form_mesh(vertices, faces)

        mesh.add_attribute("vertex_mean_curvature")
        H = mesh.get_attribute("vertex_mean_curvature")
        mean_H = H[0]

        mesh.add_attribute("vertex_gaussian_curvature")
        K = mesh.get_attribute("vertex_gaussian_curvature")
        K = K[0]

        elem = np.square(mean_H) - K
        elem = np.array(elem)

        k1 = mean_H + np.sqrt(elem)
        k2 = mean_H - np.sqrt(elem)

        si = (k1 + k2) / (k1 - k2)
        si = np.arctan(si) * (2 / np.pi)

        curvatures[idx] = si

    return curvatures

# Function to update B-factor with curvature values
def select_and_update(structure, curvatures):
    for model in structure:
        for chain in model:
            if chain.id == 'C':
                for i, residue in enumerate(chain):
                    for j, atom in enumerate(residue):
                        if i * len(residue) + j < len(curvatures):
                            curvature_value = curvatures[i * len(residue) + j]
                            if curvature_value is not None:
                                atom.set_bfactor(curvature_value)

# Initialize a PDB parser
parser = PDBParser(QUIET=True)

# Load the structure
pdb_file_path = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_surface_index_yuhao/1a2y_B_A_C_centered.pdb'
structure = parser.get_structure('1a2y_B_A_C', pdb_file_path)

# Calculate curvatures
n_neighbors_list = [7]
for n_neighbors in n_neighbors_list:
    curvatures = calculate_curvature(pdb_file_path, n_neighbors=n_neighbors)

    # Create a new PDBIO object for saving the modified structure
    io = PDBIO()

    # Save the updated chain C structure
    output_path = f'/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_surface_index_yuhao/1a2y_B_A_C_chain_C_with_curvature_{n_neighbors}.pdb'
    io.set_structure(structure)
    select_and_update(structure, curvatures)
    io.save(output_path)


In [ ]:
#@title TEST generate mesh for all the atoms of pdb
import pandas as pd
import numpy as np
import pymesh
from Bio.PDB import PDBParser, PDBIO

def calculate_curvature(pdb_file_path, n_neighbors=3):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('protein', pdb_file_path)
    model = structure[0]
    chain = model['C']

    atoms = [atom for atom in chain.get_atoms()]
    coords = np.array([atom.coord for atom in atoms])
    curvatures = [None] * len(atoms)

    for idx, atom in enumerate(atoms):
        atom_coords = np.array(coords[idx])
        dist_coords = coords - atom_coords

        # Calculate distances and get indices of nearest neighbors
        dists = np.linalg.norm(dist_coords, axis=1)
        nearest_neighbors_idx = np.argsort(dists)[1:(n_neighbors + 1)]

        # Create vertices and faces
        vertices = dist_coords[np.concatenate([[idx], nearest_neighbors_idx]), :]

        faces = [[0, i, i + 1] for i in range(1, n_neighbors)] + [[0, n_neighbors, 1]]


        faces = np.array(faces)

        # Create mesh
        mesh = pymesh.form_mesh(vertices, faces)

        mesh.add_attribute("vertex_mean_curvature")
        H = mesh.get_attribute("vertex_mean_curvature")
        mean_H = H[0]

        mesh.add_attribute("vertex_gaussian_curvature")
        K = mesh.get_attribute("vertex_gaussian_curvature")
        K = K[0]

        elem = np.square(mean_H) - K
        elem = np.array(elem)

        k1 = mean_H + np.sqrt(elem)
        k2 = mean_H - np.sqrt(elem)

        si = (k1 + k2) / (k1 - k2)
        si = np.arctan(si) * (2 / np.pi)

        curvatures[idx] = si

    return curvatures

# Function to update B-factor with curvature values
def select_and_update(structure, curvatures):
    for model in structure:
        for chain in model:
            if chain.id == 'C':
                for i, residue in enumerate(chain):
                    for j, atom in enumerate(residue):
                        if i * len(residue) + j < len(curvatures):
                            curvature_value = curvatures[i * len(residue) + j]
                            if curvature_value is not None:
                                atom.set_bfactor(curvature_value)

# Initialize a PDB parser
parser = PDBParser(QUIET=True)

# Load the structure
pdb_file_path = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_surface_index_yuhao/1a2y_B_A_C_centered.pdb'
structure = parser.get_structure('1a2y_B_A_C', pdb_file_path)

# Calculate curvatures
n_neighbors_list = [7]
for n_neighbors in n_neighbors_list:
    curvatures = calculate_curvature(pdb_file_path, n_neighbors=n_neighbors)

    # Create a new PDBIO object for saving the modified structure
    io = PDBIO()

    # Save the updated chain C structure
    output_path = f'/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_surface_index_yuhao/1a2y_B_A_C_chain_C_with_curvature_{n_neighbors}.pdb'
    io.set_structure(structure)
    select_and_update(structure, curvatures)
    io.save(output_path)


In [ ]:
#@title Read the data frame from csv file
save_dir = './drive/MyDrive/@antibody/antigen_epitope/data/antigen_surface_index_yuhao/'
df = pd.read_csv(os.path.join(save_dir, '1a2y_B_A_C_info.tsv'),delimiter='\t')

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
df

In [ ]:
#@title curvature from info.dat
# test the curvature calculation on a single protein
import pymesh

# Filter out surface residues
surface_residues = df[df['SaSaRate'] > 0.2]

# Initialize an empty list to store curvature
curvatures = [None] * len(df)

# Loop over each surface residue to find its 3 nearest neighbors
for idx, residue in surface_residues.iterrows():
    coords = df[['SCx', 'SCy', 'SCz']].values.astype(float)
    residue_coords = np.array(residue[['SCx', 'SCy', 'SCz']].values, dtype=float)
    dists = np.linalg.norm(coords - residue_coords, axis=1)
    nearest_neighbors_idx = np.argsort(dists)[1:4]  # Exclude itself

    # Create vertices (coordinates of the residue and its neighbors)
    vertices = coords[np.concatenate([[idx], nearest_neighbors_idx]), :]

    # Create faces (assuming the points form a tetrahedron)
    faces = np.array([[0, 1, 2], [0, 2, 3], [0, 3, 1], [1, 2, 3]])

    # Create mesh in PyMesh
    mesh = pymesh.form_mesh(vertices, faces)

    # Calculate Gaussian curvature
    mesh.add_attribute("vertex_gaussian_curvature")
    curvature = mesh.get_attribute("vertex_gaussian_curvature")

    # Store the average curvature
    curvatures[idx] = np.mean(curvature)

# Add the curvature column to the DataFrame
df['curvatures'] = curvatures

In [ ]:
# Normalize the 'curvatures' column (excluding NaN values)
df['curvatures'] = (df['curvatures'] - df['curvatures'].min(skipna=True)) / (df['curvatures'].max(skipna=True) - df['curvatures'].min(skipna=True))
df

In [ ]:
# Initialize a PDB parser
parser = PDB.PDBParser(QUIET=True)

# Load the structure
pdb_file_path = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_surface_index_yuhao/1a2y_B_A_C.pdb'
structure = parser.get_structure('1a2y_B_A_C', pdb_file_path)

# Create a dictionary with ResID as key and curvature as value
curvature_dict = dict(zip(df['ResID'], df['curvatures']))

# Iterate through residues and update b-factors
for model in structure:
    for chain in model:
        for residue in chain:
            res_id = residue.id[1]  # Get residue number
            if res_id in curvature_dict:
                curvature_value = curvature_dict[res_id]
                if not pd.isna(curvature_value):  # Check if curvature value is NaN
                    for atom in residue:
                        atom.set_bfactor(curvature_value)

# Save the modified structure to a new file
io = PDB.PDBIO()
io.set_structure(structure)
io.save('/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_surface_index_yuhao/1a2y_B_A_C_modified.pdb')